# Classification Metrics: Precision, Recall & F1 Score

## Learning Objectives
- Understand **Precision**, **Recall**, and **F1 Score** intuitively
- Train **Decision Tree** and **Random Forest** classifiers
- Evaluate models using the full metrics suite
- Use **PyTorch Lightning** for a neural network baseline comparison

---

## The Core Concepts

| Term | Plain English | Formula |
|------|---------------|----------|
| **Precision** | Of all the positives I *predicted*, how many were actually positive? | TP / (TP + FP) |
| **Recall** | Of all the *actual* positives, how many did I find? | TP / (TP + FN) |
| **F1 Score** | Harmonic mean of Precision & Recall — balances both | 2 × (P × R) / (P + R) |

> **Rule of thumb**: High precision = few false alarms. High recall = few misses. F1 punishes extreme imbalance between them.

## Step 1 — Install Dependencies

On Lightning AI Studio, run this cell once:

In [ ]:
# Uncomment and run on first use in Lightning AI Studio
# !pip install scikit-learn lightning torch torchmetrics matplotlib seaborn pandas

## Step 2 — Data Source: Breast Cancer Wisconsin Dataset

**Why this dataset?**
- Built into scikit-learn — no download needed
- Binary classification: malignant (1) vs benign (0)
- 569 samples, 30 numeric features
- Class imbalance makes precision/recall/F1 **actually meaningful** (unlike perfectly balanced toy data)

**Alternative datasets** (also great for learning):
- `sklearn.datasets.load_iris` — multiclass (3 flowers)
- `sklearn.datasets.fetch_openml('creditcard')` — fraud detection (extreme imbalance)
- Kaggle: [Heart Disease UCI](https://www.kaggle.com/datasets/ronitf/heart-disease-uci)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')  # 0 = malignant, 1 = benign

print(f"Dataset shape: {X.shape}")
print(f"\nClass distribution:")
print(y.value_counts().rename({0: 'Malignant', 1: 'Benign'}))
print(f"\nFeatures: {list(X.columns[:5])} ... (30 total)")
X.head(3)

In [ ]:
# Train/test split (stratified to preserve class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (important for neural net; doesn't affect trees)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

## Step 3 — Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score
)

# Train
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

# Predict
y_pred_dt = dt.predict(X_test)

# Metrics
print("=== Decision Tree ===")
print(f"Precision : {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred_dt):.4f}")
print()
print(classification_report(y_test, y_pred_dt, target_names=['Malignant', 'Benign']))

## Step 4 — Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

# Predict
y_pred_rf = rf.predict(X_test)

# Metrics
print("=== Random Forest ===")
print(f"Precision : {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred_rf):.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=['Malignant', 'Benign']))

## Step 5 — Confusion Matrix Visualization

The confusion matrix is where **TP, FP, FN, TN** live — the raw ingredients for all three metrics.

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Malignant', 'Benign'],
        yticklabels=['Malignant', 'Benign']
    )
    plt.title(title)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()

    tn, fp, fn, tp = cm.ravel()
    print(f"  TP={tp}  FP={fp}")
    print(f"  FN={fn}  TN={tn}")
    print(f"  Precision = {tp}/{tp+fp} = {tp/(tp+fp):.3f}  ← how clean are my positive predictions?")
    print(f"  Recall    = {tp}/{tp+fn} = {tp/(tp+fn):.3f}  ← how many positives did I catch?")

plot_confusion_matrix(y_test, y_pred_dt, "Decision Tree")
plot_confusion_matrix(y_test, y_pred_rf, "Random Forest")

## Step 6 — Precision-Recall Curve

Changing the **decision threshold** (default 0.5) trades precision for recall. This curve shows all possible trade-offs.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

# Get probability scores
y_proba_dt = dt.predict_proba(X_test)[:, 1]
y_proba_rf = rf.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(7, 5))

for y_proba, label, color in [
    (y_proba_dt, 'Decision Tree', 'steelblue'),
    (y_proba_rf, 'Random Forest', 'darkorange'),
]:
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    ax.plot(recall, precision, label=f"{label} (AP={ap:.3f})", color=color, lw=2)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("AP (Average Precision) = area under the PR curve. Closer to 1.0 = better.")

## Step 7 — Feature Importance (Random Forest)

Random forests give us free feature importance scores — useful for understanding what drives predictions.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=data.feature_names)
top10 = importances.nlargest(10)

plt.figure(figsize=(8, 5))
top10.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Step 8 — PyTorch Lightning Neural Network (Baseline)

Lightning AI is built on **PyTorch Lightning**. Here we replicate the same binary classification task with a simple MLP to compare results and see how torchmetrics computes F1/precision/recall.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import lightning as L
from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryF1Score

# Convert to tensors
X_tr = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr = torch.tensor(y_train.values,  dtype=torch.float32)
X_te = torch.tensor(X_test_scaled,   dtype=torch.float32)
y_te = torch.tensor(y_test.values,   dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_te, y_te), batch_size=64)

print("Tensors ready.")

In [ ]:
class CancerClassifier(L.LightningModule):
    """Simple MLP for binary classification with Lightning."""

    def __init__(self, input_dim=30, hidden_dim=64, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.loss_fn = nn.BCEWithLogitsLoss()

        # torchmetrics — these accumulate over batches automatically
        self.val_precision = BinaryPrecision()
        self.val_recall    = BinaryRecall()
        self.val_f1        = BinaryF1Score()

    def forward(self, x):
        return self.net(x).squeeze(1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss   = self.loss_fn(logits, y)
        preds  = torch.sigmoid(logits)

        self.val_precision(preds, y.int())
        self.val_recall(preds, y.int())
        self.val_f1(preds, y.int())

        self.log('val_loss',      loss,              prog_bar=True)
        self.log('val_precision', self.val_precision, prog_bar=True)
        self.log('val_recall',    self.val_recall,    prog_bar=True)
        self.log('val_f1',        self.val_f1,        prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

print("Model defined.")

In [ ]:
# Train with Lightning
model   = CancerClassifier(input_dim=30, hidden_dim=64, lr=1e-3)
trainer = L.Trainer(
    max_epochs=30,
    log_every_n_steps=5,
    enable_checkpointing=False,
    logger=False,         # Set to True + CSVLogger for full logging in Lightning AI
)
trainer.fit(model, train_loader, val_loader)

In [ ]:
# Final validation metrics
results = trainer.validate(model, val_loader, verbose=False)
r = results[0]
print("=== Lightning MLP ===")
print(f"Precision : {r['val_precision']:.4f}")
print(f"Recall    : {r['val_recall']:.4f}")
print(f"F1 Score  : {r['val_f1']:.4f}")

## Step 9 — Side-by-Side Comparison

In [ ]:
summary = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'MLP (Lightning)'],
    'Precision': [
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_rf),
        r['val_precision'],
    ],
    'Recall': [
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_rf),
        r['val_recall'],
    ],
    'F1 Score': [
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf),
        r['val_f1'],
    ],
}).set_index('Model')

print(summary.round(4).to_string())

summary.plot(kind='bar', figsize=(9, 5), rot=0, colormap='Set2', edgecolor='black', alpha=0.85)
plt.title('Precision / Recall / F1 by Model')
plt.ylabel('Score')
plt.ylim(0.8, 1.02)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Key Takeaways

| Metric | When to prioritize |
|--------|--------------|
| **Precision** | When false positives are costly (e.g., spam filter — don't flag real emails) |
| **Recall** | When false negatives are costly (e.g., cancer screening — don't miss disease) |
| **F1** | When you need a single balanced number (especially with class imbalance) |
| **PR Curve** | Full picture of precision/recall trade-off at every threshold |

**Next steps to ask about:**
- ROC-AUC vs PR-AUC and when to use each
- Cross-validation with stratified k-folds
- Hyperparameter tuning (GridSearchCV / Optuna)
- Handling severe class imbalance (SMOTE, class weights)
- Exporting and serving models from Lightning AI